# 05b — Optimized Paper Model (Ghost + MPCA + SIoU, tuned to chase 0.786)

Same **architecture** as notebook 05 (the paper's Ghost+MPCA+SIoU model) — we only stack the same **accuracy-oriented training/inference techniques** used for the optimized baseline (`updated_03`) to chase the paper's improved **mAP@0.5 = 0.786**.

| Technique | Why |
|---|---|
| **imgsz 640 → 800** | more resolution for small/low-contrast defects (crazing, rolled-in_scale) |
| **SGD + cosine LR, 200 epochs, patience 60** | train the from-scratch Ghost backbone fully to convergence |
| **close_mosaic=20, mixup=0.1** | clean final epochs sharpen localization; mild regularization |
| **TTA (augment=True) at eval** | flip/scale ensembling at inference — free accuracy |
| **NMS IoU=0.6** | the paper's dynamic-NMS tweak for dense small targets |

> *Separate experiment* from the fair 3-way comparison (nb 03/05/06) — heavier recipe. Results → `results/improved_opt/`.
> ⚠️ imgsz=800 is slow (~4–5 h / 200 ep). Set `IMGSZ=640, BATCH=16` in the train cell for ~3× faster.

## 1. Check environment
> **Kernel:** the venv with Ultralytics — `C:\Users\student\Downloads\files\.venv` (`.venv (Python 3.10.8)`).

In [1]:
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

PyTorch: 2.6.0+cu124 | CUDA: True
GPU: NVIDIA RTX 2000 Ada Generation


## 2. Register custom modules & locate dataset
`MPCA`/`SIoU` are added by this project under `src/modules/`. `register()` binds the `MPCA` layer name and patches the box loss to SIoU — run it **before** building the model, and it must stay active to unpickle `best.pt` at eval.

In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))            # make 'src' importable

from src.modules import register
register()                                # activates MPCA + SIoU

DATA_CFG = ROOT / 'data' / 'neu-det-yolo' / 'data.yaml'
assert DATA_CFG.exists(), 'Run 01_data_preparation.ipynb first!'
CLASSES = ['crazing','inclusion','patches','pitted_surface','rolled-in_scale','scratches']
print('Data config:', DATA_CFG)

[src.modules.register] activated: MPCA (end-of-backbone attention), SIoU regression loss
Data config: c:\Users\student\Desktop\SteelDefectDetection\data\neu-det-yolo\data.yaml


## 3. Build the model & transfer COCO weights
Build from `configs/yolov8n_improved.yaml`, then transfer matching COCO weights from `yolov8n.pt`. The from-scratch parts train fresh; standard layers transfer.

In [3]:
from ultralytics import YOLO

CFG = ROOT / 'configs' / 'yolov8n_improved.yaml'
model = YOLO(str(CFG))                    # build architecture from YAML

PRETRAINED = ROOT / 'yolov8n.pt'
model.load(str(PRETRAINED) if PRETRAINED.exists() else 'yolov8n.pt')   # transfer COCO weights
model.info()                              # ~2.4M params / 6.4 GFLOPs (baseline 3.0M / 8.1)

Transferred 26/521 items from pretrained weights
YOLOv8n_improved summary: 215 layers, 2,396,494 parameters, 2,396,478 gradients, 6.3 GFLOPs


(215, 2396494, 2396478, 6.347392000000001)

## 4. Train (optimized recipe)
Same accuracy stack as the optimized baseline (`updated_03`): imgsz=800, 200 ep, SGD+cosine, close_mosaic=20, mixup=0.1. `workers=0` is Windows-safe.
> **VRAM:** imgsz=800 + batch=8 fits ~16 GB. Ghost backbone is light — batch=8 is comfortable; drop to 4 only if OOM.

In [4]:
IMGSZ = 800              # <- main lever; set 640 for ~3x faster training
BATCH = 8 if DEVICE == 0 else 2    # 800px+16GB; drop to 4 if OOM (or 16 if IMGSZ=640)

results = model.train(
    data=str(DATA_CFG),
    epochs=200,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    cache=True,
    workers=0,            # Windows-safe (avoids close_mosaic deadlock)
    project=str(ROOT / 'results'),
    name='improved_opt',
    exist_ok=True,
    optimizer='SGD',      # SGD+momentum (lr0=0.01)
    cos_lr=True,          # cosine LR decay
    patience=60,          # generous: let the from-scratch backbone converge
    close_mosaic=20,      # last 20 epochs on clean (non-mosaic) images
    mixup=0.1,            # mild regularization for the small dataset
    seed=42,
    plots=True,
)
print('Training done. Best weights:', ROOT / 'results/improved_opt/weights/best.pt')

New https://pypi.org/project/ultralytics/8.4.64 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.10.8 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=20, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=c:\Users\student\Desktop\SteelDefectDetection\data\neu-det-yolo\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=c:\User

## 5. Evaluate — ablation: plain vs optimized (TTA + NMS 0.6) on the TEST set
Both reload `best.pt` so this runs without re-training (the `register*()` call above must be active so custom layers unpickle). Target: **paper improved target: 0.786**.

In [5]:
best = ROOT / 'results' / 'improved_opt' / 'weights' / 'best.pt'   # register*() above lets this unpickle

plain = YOLO(str(best)).val(data=str(DATA_CFG), split='test', verbose=False)
opt   = YOLO(str(best)).val(data=str(DATA_CFG), split='test', augment=True, iou=0.6, verbose=False)

print(f"{'eval':<22}{'mAP@0.5':>10}{'mAP@.5:.95':>12}{'P':>8}{'R':>8}")
print(f"{'plain':<22}{plain.box.map50:>10.4f}{plain.box.map:>12.4f}{plain.box.mp:>8.3f}{plain.box.mr:>8.3f}")
print(f"{'optimized (TTA+NMS.6)':<22}{opt.box.map50:>10.4f}{opt.box.map:>12.4f}{opt.box.mp:>8.3f}{opt.box.mr:>8.3f}")
print(f"\npaper improved target: 0.786   |   best here: {max(plain.box.map50, opt.box.map50):.4f}")

Ultralytics 8.4.51  Python-3.10.8 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
YOLOv8n_improved summary (fused): 138 layers, 2,391,462 parameters, 0 gradients, 6.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 38.518.0 MB/s, size: 16.8 KB)
val: Scanning C:\Users\student\Desktop\SteelDefectDetection\data\neu-det-yolo\labels\test.cache... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 3.1it/s 3.8s0.1s
                   all        180        413      0.671      0.701      0.715      0.378
Speed: 2.5ms preprocess, 14.0ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to C:\Users\student\Desktop\SteelDefectDetection\notebooks\runs\detect\val-30
Ultralytics 8.4.51  Python-3.10.8 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
YOLOv8n_improved summary (fused): 138 layers, 2,391,462 parame

## 6. Per-class table (best eval) & save summary
Writes `results/improved_opt/metrics_summary.txt`.

In [ ]:
import pandas as pd
best_res = opt if float(opt.box.map50) >= float(plain.box.map50) else plain
tag = 'optimized (TTA+NMS0.6)' if best_res is opt else 'plain'

rows = [{'class': n, 'mAP@0.5': round(float(best_res.box.ap50[i]), 4),
         'mAP@0.5:0.95': round(float(best_res.box.ap[i]), 4)} for i, n in enumerate(CLASSES)]
df = pd.DataFrame(rows)
df.loc[len(df)] = ['ALL (mean)', round(float(best_res.box.map50), 4), round(float(best_res.box.map), 4)]

run_dir = ROOT / 'results' / 'improved_opt'; run_dir.mkdir(parents=True, exist_ok=True)
summary = run_dir / 'metrics_summary.txt'
with open(summary, 'w') as f:
    f.write('Optimized Paper Model YOLOv8n (Ghost+MPCA+SIoU, imgsz=800, SGD, 200ep, TTA+NMS) - TEST set\n')
    f.write('=' * 70 + '\n')
    f.write(f'best eval    : {tag}\n')
    f.write(f'mAP@0.5      : {best_res.box.map50:.4f}  (paper improved target: 0.786)\n')
    f.write(f'mAP@0.5:0.95 : {best_res.box.map:.4f}\n')
    f.write(f'precision    : {best_res.box.mp:.4f}\n')
    f.write(f'recall       : {best_res.box.mr:.4f}\n\nPer-class mAP@0.5:\n')
    for i, n in enumerate(CLASSES):
        f.write(f'  {n:<18}{float(best_res.box.ap50[i]):.4f}\n')
print('saved', summary)
df

saved c:\Users\student\Desktop\SteelDefectDetection\results\improved_opt\metrics_summary.txt


,class,mAP@0.5,mAP@0.5:0.95
0,crazing,0.4486,0.1836
1,inclusion,0.8180,0.4217
2,patches,0.9139,0.5817
3,pitted_surface,0.8540,0.4206
4,rolled-in_scale,0.5469,0.2134
5,scratches,0.8351,0.3700
6,ALL (mean),0.7361,0.3652


: 

✅ **Optimized paper model done.** Weights → `results/improved_opt/weights/best.pt`, metrics → `results/improved_opt/metrics_summary.txt`. Compare the best test mAP@0.5 to the paper's **0.786** and to the fair-recipe run (notebook 05). Next levers if short: `IMGSZ=960` or `epochs=300`.